# Noise-ceiling reliability: `mean` vs `simple` vs `evidence`

Compares the three behavioral-RDM aggregation methods (see `analysis/rdms/behavioral.py`) via Nili et al. (2014)-style noise-ceiling bounds, computed by `analysis/behavioral/reliability.py`:

- **Lower bound**: leave-one-participant-out (exact, N=79/76 folds).
- **Upper bound**: split-half (S=100 random draws, same splits reused across all 3 aggregators).
- **Metrics**: Kendall's tau (leading) and Spearman's rho (secondary), computed directly on condensed RDM vectors over jointly-covered pairs. Procrustes/MDS was dropped entirely -- every participant's own trial-derived pair graph is disconnected into ~18 components (confirmed against real data), so a single held-out participant's data can never be MDS-embedded regardless of aggregation method.

This notebook loads/builds the tidy reliability data, runs the pre-planned pairwise Wilcoxon signed-rank comparisons (evidence-vs-mean primary; evidence-vs-simple, simple-vs-mean secondary), and renders the summary figure. See the plan (`purring-mixing-horizon.md`, section 6) for the full methodology and the decisions behind it.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import wilcoxon

from analysis.behavioral.reliability import run_comparison, RESULTS_DIR, _DEFAULT_CACHE_DIR

_TEMPLATE = "plotly_white"  # matches analysis/rdms/imagenet_vs_path.py & hierarchy_comparison.py

## Load or build the reliability data

Reuses the cached CSV under `analysis/results/behavioral/` if `reliability.py`'s CLI has already been run for both cohorts (a full S=100 split-half + N=79/76 LOO sweep over 3 aggregators is nontrivial to re-run -- see the plan's compute-cost notes); otherwise runs it here. Point `cache_dir` at the main checkout's `analysis/results/parsed_data/` if running this notebook from a worktree (that cache isn't copied into worktrees).

In [2]:
cache_dir = _DEFAULT_CACHE_DIR  # override if running from a worktree, e.g. the main checkout's analysis/results/parsed_data
csv_path = RESULTS_DIR / "reliability.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    df = pd.concat([
        run_comparison("pre_shine", cache_dir=cache_dir, n_splits=100, seed=0),
        run_comparison("post_shine", cache_dir=cache_dir, n_splits=100, seed=0),
    ], ignore_index=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)

df = df[~df["underpowered"]]  # drop any fold that didn't clear min_overlap jointly-covered pairs
df.head()

,cohort,aggregation,bound,fold_id,n_overlap,underpowered,metric,value
0,pre,mean,lower,6998a40984a21b26ed35074c,2129,False,kendall,0.089705
1,pre,mean,lower,68598a1d4cebd213b2abb1d9,2240,False,kendall,0.130174
2,pre,mean,lower,67341b607b7ab78bc222464e,2215,False,kendall,0.144703
3,pre,mean,lower,69a50d29b53acc0ed832d33f,2205,False,kendall,0.083020
4,pre,mean,lower,65fbccca15536d60943e0e40,2178,False,kendall,0.190198


## Summary table: mean ± SEM per (aggregator × metric × cohort × bound)

In [3]:
summary = (
    df.groupby(["cohort", "bound", "metric", "aggregation"])["value"]
    .agg(mean="mean", sem=lambda s: s.std(ddof=1) / np.sqrt(len(s)), n="count")
    .reset_index()
)
summary

,cohort,bound,metric,aggregation,mean,sem,n
0,post,lower,kendall,evidence,0.125402,0.003778,76
1,post,lower,kendall,mean,0.137639,0.004278,76
2,post,lower,kendall,simple,0.138984,0.004299,76
3,post,lower,spearman,evidence,0.187049,0.005577,76
4,post,lower,spearman,mean,0.205116,0.006321,76
5,post,lower,spearman,simple,0.207041,0.006348,76
6,post,upper,kendall,evidence,0.119321,0.000234,100
7,post,upper,kendall,mean,0.138662,0.000254,100
8,post,upper,kendall,simple,0.140392,0.000252,100
9,post,upper,spearman,evidence,0.178258,0.000339,100


## Pairwise Wilcoxon signed-rank comparisons

Pre-planned pairwise comparisons per (cohort, bound, metric) group, on matched fold/split indices (paired by `fold_id`, which corresponds to the same held-out participant for the lower bound, or the same shared split for the upper bound -- see `reliability.py`'s `make_splits()`): **evidence-vs-mean (primary)**, evidence-vs-simple, simple-vs-mean (secondary -- decompose whether rescaling alone (`simple`) or rescaling + magnitude-weighting (`evidence`) drives any difference). Effect size is the matched-pairs rank-biserial correlation. Kendall's tau is the leading metric for the "is `evidence` best" verdict; Spearman is reported alongside as a robustness check.

In [4]:
_COMPARISONS = [("evidence", "mean"), ("evidence", "simple"), ("simple", "mean")]


def _wilcoxon_row(x: np.ndarray, y: np.ndarray) -> dict:
    diff = np.asarray(x) - np.asarray(y)
    diff_nz = diff[diff != 0]
    n = len(diff_nz)
    if n == 0:
        return {"n": 0, "statistic": np.nan, "p_value": np.nan, "effect_size": np.nan}
    stat, p = wilcoxon(x, y)
    ranks = pd.Series(np.abs(diff_nz)).rank().to_numpy()
    r_plus = ranks[diff_nz > 0].sum()
    r_minus = ranks[diff_nz < 0].sum()
    effect_size = (r_plus - r_minus) / ranks.sum()
    return {"n": n, "statistic": float(stat), "p_value": float(p), "effect_size": float(effect_size)}


rows = []
for (cohort, bound, metric), group in df.groupby(["cohort", "bound", "metric"]):
    wide = group.pivot(index="fold_id", columns="aggregation", values="value").dropna()
    for a, b in _COMPARISONS:
        if a not in wide.columns or b not in wide.columns:
            continue
        result = _wilcoxon_row(wide[a].to_numpy(), wide[b].to_numpy())
        rows.append({"cohort": cohort, "bound": bound, "metric": metric, "comparison": f"{a}_vs_{b}", **result})

wilcoxon_results = pd.DataFrame(rows)
wilcoxon_results

,cohort,bound,metric,comparison,n,statistic,p_value,effect_size
0,post,lower,kendall,evidence_vs_mean,76,79.0,7.752802e-13,-0.946001
1,post,lower,kendall,evidence_vs_simple,76,60.0,3.762927e-13,-0.958988
2,post,lower,kendall,simple_vs_mean,76,938.0,6.565598e-03,0.358852
3,post,lower,spearman,evidence_vs_mean,76,68.0,5.107536e-13,-0.953520
4,post,lower,spearman,evidence_vs_simple,76,59.0,3.621501e-13,-0.959672
5,post,lower,spearman,simple_vs_mean,76,933.0,6.069611e-03,0.362269
6,post,upper,kendall,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000
7,post,upper,kendall,evidence_vs_simple,100,0.0,3.896560e-18,-1.000000
8,post,upper,kendall,simple_vs_mean,100,114.0,1.134379e-16,0.954851
9,post,upper,spearman,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000


In [5]:
# Primary claim: is 'evidence' best, per the leading metric (Kendall), at both bounds?
wilcoxon_results.query("metric == 'kendall' and comparison == 'evidence_vs_mean'")

,cohort,bound,metric,comparison,n,statistic,p_value,effect_size
0,post,lower,kendall,evidence_vs_mean,76,79.0,7.752802e-13,-0.946001
6,post,upper,kendall,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000
12,pre,lower,kendall,evidence_vs_mean,79,32.0,3.873170e-14,-0.979747
18,pre,upper,kendall,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000


## Figure: reliability by aggregator, bound, metric, and cohort

2×2 grid (rows = metric, cols = cohort), each subplot a grouped bar chart (x = bound, grouped/colored by aggregator), bar height = mean, error bars = SEM.

In [6]:
_METRICS = ["kendall", "spearman"]
_COHORTS = ["pre", "post"]
_AGGREGATIONS = ["mean", "simple", "evidence"]
_BOUNDS = ["lower", "upper"]
_COLORS = {"mean": "#636EFA", "simple": "#EF553B", "evidence": "#00CC96"}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{metric.title()} — {cohort}-SHINE" for metric in _METRICS for cohort in _COHORTS],
    shared_yaxes=True,
)

for row, metric in enumerate(_METRICS, start=1):
    for col, cohort in enumerate(_COHORTS, start=1):
        cell = summary[(summary["metric"] == metric) & (summary["cohort"] == cohort)]
        for aggregation in _AGGREGATIONS:
            sub = cell[cell["aggregation"] == aggregation].set_index("bound").reindex(_BOUNDS)
            fig.add_trace(
                go.Bar(
                    x=_BOUNDS, y=sub["mean"], error_y=dict(type="data", array=sub["sem"]),
                    name=aggregation, marker_color=_COLORS[aggregation],
                    legendgroup=aggregation, showlegend=(row == 1 and col == 1),
                ),
                row=row, col=col,
            )

fig.update_layout(
    title="Noise-ceiling reliability: mean vs simple vs evidence"
          "<br><sub>lower bound = leave-one-out; upper bound = split-half (S=100)</sub>",
    barmode="group", template=_TEMPLATE, height=760, width=980,
    legend=dict(orientation="h", y=-0.08, x=0.5, xanchor="center"),
)
fig.show()